## Explicit vs Implicit Feedback in Recommendation Systems

### **Explicit Feedback**
- **Definition**: Users provide direct ratings or scores for items (e.g rate all items.- **Definition**: Users provide direct ratings or scores for items (e.g., 1–5 stars).
- **Example**: Movie ratings in the MovieLens dataset.
- **ALS Setting**: `implicitPrefs=False`


### **Implicit Feedback**
- **Definition**: Preferences inferred from user behavior rather than direct ratings.
- **Examples**:
  - Clicks, views, purchases, watch time.
- **Characteristics**:
  - Data is **abundant** but **noisy** (not all interactions mean positive preference).
  - Requires interpreting signals (e.g., a click might not mean the user liked the item).
- **ALS Setting**: `implicitPrefs=True`


### **Key Differences**
| Aspect           | Explicit Feedback        | Implicit Feedback          |
|------------------|-------------------------|----------------------------|
| Source           | Direct ratings          | Behavioral signals         |
| Accuracy         | High                   | Lower (inferred)          |
| Data Volume      | Sparse                 | Dense                      |
| ALS Parameter    | `implicitPrefs=False`  | `implicitPrefs=True`       |



**Key Takeaway**
- Explicit feedback models often perform better when ratings are available.
- Implicit feedback models are useful when ratings are missing but interaction data exists.
- In some situations, we may want to compare how implicit vs explicit assumptions affect recommendations.



**In this exercise, we will build two recommendation systems from pyspark to deal with the whole dataset, not only the records with rating more than 4**

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:

import pandas as pd
# Import Spark ALS and evaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Load data into Spark DataFrame
# Load data

file_path = '/Volumes/movies_data/default/all_mvoies_data_100k/u.data' #change for your path
user_movie_data = pd.read_csv(file_path, sep='\t', names=['userId', 'movieId', 'rating', 'timestamp'])


# Create Spark DataFrame
ratings_df = spark.createDataFrame(user_movie_data)
ratings_df.show(5)


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|   196|    242|     3|881250949|
|   186|    302|     3|891717742|
|    22|    377|     1|878887116|
|   244|     51|     2|880606923|
|   166|    346|     1|886397596|
+------+-------+------+---------+
only showing top 5 rows


In [0]:
#splitting the data and buidling the evaluation function
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Split data
train_df, test_df = ratings_df.randomSplit([0.8, 0.2], seed=42)

# Evaluator
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

In [0]:
# Model 1: ALS Explicit
als_explicit = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
                   implicitPrefs=False, rank=10, maxIter=10, regParam=0.1,
                   coldStartStrategy="drop")
model_explicit = als_explicit.fit(train_df)
predictions_explicit = model_explicit.transform(test_df)
rmse_explicit = evaluator.evaluate(predictions_explicit)

In [0]:
# Model 2: ALS Implicit
als_implicit = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
                   implicitPrefs=True, rank=10, maxIter=10, regParam=0.1,
                   coldStartStrategy="drop")
model_implicit = als_implicit.fit(train_df)
predictions_implicit = model_implicit.transform(test_df)
rmse_implicit = evaluator.evaluate(predictions_implicit)

In [0]:
print(f"RMSE Explicit: {rmse_explicit}")
print(f"RMSE Implicit: {rmse_implicit}")

RMSE Explicit: 0.9229189664521714
RMSE Implicit: 3.1557215199630892


**Task 2, find the top five movies simialr to movie_id 10 and the top 5 users similar to user_id 100 using the two models**

**Provide your results in a good visualization format**

In [0]:
# add this cell to the M8_Pyspark_recommendation_system notebook after building the two models
import pandas as pd
import numpy as np

# Extract factors from ALS models
item_factors_explicit = model_explicit.itemFactors
user_factors_explicit = model_explicit.userFactors

item_factors_implicit = model_implicit.itemFactors
user_factors_implicit = model_implicit.userFactors

# Convert to dictionaries
item_vecs_explicit = {row.id: np.array(row.features) for row in item_factors_explicit.collect()}
user_vecs_explicit = {row.id: np.array(row.features) for row in user_factors_explicit.collect()}

item_vecs_implicit = {row.id: np.array(row.features) for row in item_factors_implicit.collect()}
user_vecs_implicit = {row.id: np.array(row.features) for row in user_factors_implicit.collect()}

# Cosine similarity function
def cosine_sim(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Target IDs
target_movie_id = 10
target_user_id = 100

# Compute similarities
similar_movies_explicit = sorted([(mid, cosine_sim(item_vecs_explicit[target_movie_id], vec)) 
                                  for mid, vec in item_vecs_explicit.items() if mid != target_movie_id],
                                  key=lambda x: x[1], reverse=True)[:5]

similar_movies_implicit = sorted([(mid, cosine_sim(item_vecs_implicit[target_movie_id], vec)) 
                                  for mid, vec in item_vecs_implicit.items() if mid != target_movie_id],
                                  key=lambda x: x[1], reverse=True)[:5]

similar_users_explicit = sorted([(uid, cosine_sim(user_vecs_explicit[target_user_id], vec)) 
                                 for uid, vec in user_vecs_explicit.items() if uid != target_user_id],
                                 key=lambda x: x[1], reverse=True)[:5]

similar_users_implicit = sorted([(uid, cosine_sim(user_vecs_implicit[target_user_id], vec)) 
                                 for uid, vec in user_vecs_implicit.items() if uid != target_user_id],
                                 key=lambda x: x[1], reverse=True)[:5]

# Convert to DataFrames for nice display
movies_df = pd.DataFrame({
    "Explicit Model": [f"Movie {m[0]} (sim={m[1]:.4f})" for m in similar_movies_explicit],
    "Implicit Model": [f"Movie {m[0]} (sim={m[1]:.4f})" for m in similar_movies_implicit]
})

users_df = pd.DataFrame({
    "Explicit Model": [f"User {u[0]} (sim={u[1]:.4f})" for u in similar_users_explicit],
    "Implicit Model": [f"User {u[0]} (sim={u[1]:.4f})" for u in similar_users_implicit]
})

print("Top 5 Similar Movies to Movie ID 10")
display(movies_df)

print("Top 5 Similar Users to User ID 100")
display(users_df)

Top 5 Similar Movies to Movie ID 10


Explicit Model,Implicit Model
Movie 921 (sim=0.9731),Movie 813 (sim=0.9273)
Movie 855 (sim=0.9590),Movie 30 (sim=0.9011)
Movie 1658 (sim=0.9374),Movie 224 (sim=0.8991)
Movie 1118 (sim=0.9329),Movie 1068 (sim=0.8990)
Movie 1573 (sim=0.9301),Movie 713 (sim=0.8846)


Top 5 Similar Users to User ID 100


Explicit Model,Implicit Model
User 690 (sim=0.9702),User 428 (sim=0.9950)
User 563 (sim=0.9289),User 206 (sim=0.9918)
User 518 (sim=0.9286),User 752 (sim=0.9893)
User 81 (sim=0.9282),User 88 (sim=0.9885)
User 782 (sim=0.9282),User 787 (sim=0.9873)
